# adcsim Kaggle — interchain disulfide repair

Rebuild the missing heavy-chain Cys223 — light-chain Cys214 disulfide
in the Rosetta-relaxed Trastuzumab hinge model `B220_235_rosetta_best.pdb`.

**Why:** the 4 interchain disulfides of a standard IgG1 are
A229–B229, A232–B232 (heavy–heavy hinge) and
A223–C214, B223–D214 (heavy–light). In the current model only 3 of 4
are formed; A223–C214 sits 4.47 A apart (not bonded). This corrupts the
per-site SASA used by the conjugation model.

**Fix:** pull the two SG atoms to the disulfide geometry (~2.02 A) with a
strong harmonic constraint, minimise only nearby sidechains, then formally
form the disulfide and re-relax a small pocket.


In [ ]:
# Install PyRosetta (CPU). Requires Internet ON and license acceptance.
%pip -q install pyrosetta-installer rdkit numpy
from pyrosetta_installer import install_pyrosetta
install_pyrosetta()
print('install done')


In [ ]:
# Locate / upload the antibody PDB and set paths.
# Upload 'B220_235_rosetta_best.pdb' to /kaggle/working/input/ then edit PDB_IN.
from pathlib import Path
import os

PDB_IN = Path('/kaggle/working/input/B220_235_rosetta_best.pdb')
PDB_OUT = Path('/kaggle/working/B220_235_disulfide_repaired.pdb')

if not PDB_IN.exists():
    raise FileNotFoundError(f"{PDB_IN} not found — upload the PDB first")
print('input PDB found:', PDB_IN.name, PDB_IN.stat().st_size, 'bytes')


In [ ]:
# Init PyRosetta ONCE (no ligand params needed — protein-only repair).
import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.id import AtomID
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.scoring import ScoreFunctionFactory, ScoreType
from pyrosetta.rosetta.core.scoring.constraints import AtomPairConstraint
from pyrosetta.rosetta.core.scoring.func import HarmonicFunc
from pyrosetta.rosetta.protocols.minimization_packing import MinMover
from pyrosetta.rosetta.protocols.relax import FastRelax

pyrosetta.init('-mute all -ignore_unrecognized_res false')
print('pyrosetta init OK')


In [ ]:
# Load pose and map the 4 target interchain disulfide pairs (pdb -> pose idx).
pose = pose_from_pdb(str(PDB_IN))
info = pose.pdb_info()

# target pairs: (chain_res) heavy Cys223 -> light Cys214, heavy-heavy hinge
PAIRS_TARGET = [('A', 223, 'C', 214), ('B', 223, 'D', 214),
                 ('A', 229, 'B', 229), ('A', 232, 'B', 232)]

def sg_atom_id(chain, resnum):
    p = info.pdb2pose(chain, resnum)
    r = pose.residue(p)
    assert r.name3() == 'CYS', f'{chain}:{resnum} is {r.name3()}, not CYS'
    assert r.has('SG'), f'{chain}:{resnum} has no SG'
    return p, AtomID(r.atom_index('SG'), p)

pairs = []
for ca, ra, cb, rb in PAIRS_TARGET:
    pa, a = sg_atom_id(ca, ra)
    pb, b = sg_atom_id(cb, rb)
    d = (pose.xyz(a) - pose.xyz(b)).norm()
    pairs.append({'ca': ca, 'ra': ra, 'cb': cb, 'rb': rb, 'pa': pa, 'pb': pb, 'd': d})

print('current interchain disulfide geometry:')
for p in pairs:
    ok = 'OK' if p['d'] < 2.5 else 'MISSING!'
    print(f"  {p['ca']}:CYS{p['ra']} - {p['cb']}:CYS{p['rb']}  {p['d']:.2f} A  {ok}")


In [ ]:
# QC-before: count how many of the 4 disulfides are actually formed (< 2.5 A).
formed = [p for p in pairs if p['d'] < 2.5]
missing = [p for p in pairs if p['d'] >= 2.5]
print(f'formed:   {len(formed)}/4')
print(f'missing:  {len(missing)}/4')
for p in missing:
    print(f"  need repair: {p['ca']}:CYS{p['ra']} - {p['cb']}:CYS{p['rb']} ({p['d']:.2f} A)")

# sanity: we expect exactly one missing pair (A223-C214)
assert len(missing) == 1, f'expected exactly 1 missing pair, got {len(missing)}'
print('\nexpectation confirmed: exactly one missing disulfide')


In [ ]:
# Form each missing disulfide: strong SG-SG harmonic + constrained minimise, then declare bond.
DISULF = 2.02   # ideal S-S bond length (A)
SIGMA = 0.02
NEIGHBOR_CUT = 8.0

def repair_disulfide(pose, pair, weight=100.0):
    pa, pb = pair['pa'], pair['pb']
    sfxn = ScoreFunctionFactory.create_score_function('ref2015')
    sfxn.set_weight(ScoreType.atom_pair_constraint, weight)

    # heavy-atom constraint pulling SG_a onto SG_b
    sg_a = AtomID(pose.residue(pa).atom_index('SG'), pa)
    sg_b = AtomID(pose.residue(pb).atom_index('SG'), pb)
    pose.add_constraint(AtomPairConstraint(sg_a, sg_b, HarmonicFunc(DISULF, SIGMA)))

    # mobile: chi of the two Cys + any residue within NEIGHBOR_CUT of either SG
    mm = MoveMap()
    mm.set_bb(False); mm.set_chi(False); mm.set_jump(False)
    mm.set_chi(pa, True); mm.set_chi(pb, True)
    anchor_a = pose.xyz(sg_a); anchor_b = pose.xyz(sg_b)
    n_nbr = 0
    for i in range(1, pose.total_residue() + 1):
        if i in (pa, pb):
            continue
        r = pose.residue(i)
        if not r.is_protein():
            continue
        xyz = r.nbr_atom_xyz()
        if (xyz - anchor_a).norm() <= NEIGHBOR_CUT or (xyz - anchor_b).norm() <= NEIGHBOR_CUT:
            mm.set_chi(i, True)
            n_nbr += 1

    mover = MinMover(mm, sfxn, 'lbfgs_armijo_nonmonotone', 0.01, True)
    mover.max_iter(200)
    before = sfxn(pose)
    mover.apply(pose)
    after = sfxn(pose)
    # clear constraints to avoid double-counting on subsequent pairs
    pose.remove_constraints()
    d_new = (pose.xyz(sg_a) - pose.xyz(sg_b)).norm()
    return {'n_nbr': n_nbr, 'score_before': before, 'score_after': after, 'd_after': d_new}

repair_log = []
for p in missing:
    print(f"repairing {p['ca']}:CYS{p['ra']} - {p['cb']}:CYS{p['rb']} ...", flush=True)
    rec = repair_disulfide(pose, p)
    repair_log.append({**p, 'n_nbr': rec['n_nbr']})
    print(f"  d {p['d']:.2f} -> {rec['d_after']:.2f} A  n_nbr={rec['n_nbr']}  "
          f"score {rec['score_before']:.1f} -> {rec['score_after']:.1f}", flush=True)


In [ ]:
# QC-after: verify every repaired SG-SG distance is now within disulfide length.
ok = True
for p in pairs:
    pa, pb = p['pa'], p['pb']
    a = AtomID(pose.residue(pa).atom_index('SG'), pa)
    b = AtomID(pose.residue(pb).atom_index('SG'), pb)
    d = (pose.xyz(a) - pose.xyz(b)).norm()
    status = 'OK' if d < 2.5 else 'STILL MISSING'
    if d >= 2.5:
        ok = False
    print(f"  {p['ca']}:CYS{p['ra']} - {p['cb']}:CYS{p['rb']}  {d:.2f} A  {status}")
assert ok, 'some disulfide still not formed — raise weight / re-run minimisation'
print('\nall 4 interchain disulfides now at disulfide geometry')


In [ ]:
# Declare the disulfide as a chemical bond (so scoring/export treat it as bonded).
# form_disulfide lives in core.conformation; guarded for API-version variance.
declared = []
for p in pairs:
    try:
        from pyrosetta.rosetta.core.conformation import form_disulfide
        form_disulfide(pose.conformation(), p['pa'], p['pb'])
        declared.append(p)
    except Exception as e:
        # fall back to a strong distance constraint (geometric bond proxy)
        print(f'  [warn] form_disulfide failed for {p["ca"]}{p["ra"]}-{p["cb"]}{p["rb"]}: {e}')
        a = AtomID(pose.residue(p['pa']).atom_index('SG'), p['pa'])
        b = AtomID(pose.residue(p['pb']).atom_index('SG'), p['pb'])
        pose.add_constraint(AtomPairConstraint(a, b, HarmonicFunc(DISULF, SIGMA)))

print(f'declared disulfide bonds: {len(declared)}/4 via form_disulfide')


In [ ]:
# Release strain around the repaired bonds: soft S-S constraint + local minimise.
# Reuses the MoveMap + MinMover API already verified in cell 6 (no residue-selector drift).
repair_res = []
for p in missing:
    repair_res.append(p['pa']); repair_res.append(p['pb'])

sfxn = ScoreFunctionFactory.create_score_function('ref2015')
sfxn.set_weight(ScoreType.atom_pair_constraint, 20.0)  # soft: keep S-S formed
for p in pairs:
    a = AtomID(pose.residue(p['pa']).atom_index('SG'), p['pa'])
    b = AtomID(pose.residue(p['pb']).atom_index('SG'), p['pb'])
    pose.add_constraint(AtomPairConstraint(a, b, HarmonicFunc(DISULF, 0.05)))

mm = MoveMap()
mm.set_bb(False); mm.set_chi(False); mm.set_jump(False)
for i in range(1, pose.total_residue() + 1):
    if i in repair_res:
        # the two repaired Cys: backbone + chi both free (tiny pocket relax)
        mm.set_bb(i, True); mm.set_chi(i, True)
        continue
    r = pose.residue(i)
    if not r.is_protein():
        continue
    xyz = r.nbr_atom_xyz()
    near = any((xyz - pose.xyz(AtomID(pose.residue(j).atom_index('SG'), j))).norm() <= NEIGHBOR_CUT
               for j in repair_res)
    if near:
        mm.set_chi(i, True)

relaxer = MinMover(mm, sfxn, 'lbfgs_armijo_nonmonotone', 0.005, True)
relaxer.max_iter(300)
before = sfxn(pose)
relaxer.apply(pose)
after = sfxn(pose)
pose.remove_constraints()
print(f'local relax: score {before:.1f} -> {after:.1f}')


In [ ]:
# Final QC: re-verify the 4 disulfides, then dump the repaired PDB.
import numpy as np
print('final disulfide check:')
final_dists = {}
for p in pairs:
    a = AtomID(pose.residue(p['pa']).atom_index('SG'), p['pa'])
    b = AtomID(pose.residue(p['pb']).atom_index('SG'), p['pb'])
    d = (pose.xyz(a) - pose.xyz(b)).norm()
    final_dists[f"{p['ca']}:CYS{p['ra']}-{p['cb']}:CYS{p['rb']}"] = round(float(d), 3)
    print(f"  {p['ca']}:CYS{p['ra']} - {p['cb']}:CYS{p['rb']}  {d:.2f} A")

pose.dump_pdb(str(PDB_OUT))
print(f'\nwrote {PDB_OUT}')

import json as _json
summary = {
    'input': PDB_IN.name,
    'output': PDB_OUT.name,
    'repaired_pairs': [p['ca'] + ':' + str(p['ra']) + '-' + p['cb'] + ':' + str(p['rb']) for p in missing],
    'final_sg_distances': final_dists,
}
Path('/kaggle/working/disulfide_repair_summary.json').write_text(_json.dumps(summary, indent=2))
print('wrote disulfide_repair_summary.json')


## Notes

- Repairs only the missing heavy–light disulfide(s); the hinge S-S pairs are untouched.
- Mobile region is limited to the two Cys plus 8 A neighbours, so the rest of the
  structure (already Phase-1 RMSD-validated) is unchanged.
- Download `B220_235_disulfide_repaired.pdb`: it becomes the new `antibody_pdb`
  in `config.yaml` and the new input to Phase 2/3.
